## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [7]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.truncated_normal([784, 512], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([512]))
        self.W2 = tf.Variable(tf.random.truncated_normal([512, 256], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([256]))
        self.W3 = tf.Variable(tf.random.truncated_normal([256, 10], stddev=0.1))
        self.b3 = tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 784])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        h2 = tf.nn.relu(tf.matmul(h1, self.W2) + self.b2)
        logits = tf.matmul(h2, self.W3) + self.b3
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [8]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.b1, model.W2, model.b2, model.W3, model.b3]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [9]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.911922 ; accuracy 0.0638
epoch 1 : loss 2.7951581 ; accuracy 0.0685
epoch 2 : loss 2.7037098 ; accuracy 0.0729
epoch 3 : loss 2.6289907 ; accuracy 0.07941667
epoch 4 : loss 2.5661044 ; accuracy 0.08625
epoch 5 : loss 2.5118103 ; accuracy 0.09543333
epoch 6 : loss 2.4638712 ; accuracy 0.104433335
epoch 7 : loss 2.420686 ; accuracy 0.11518333
epoch 8 : loss 2.3810947 ; accuracy 0.12636666
epoch 9 : loss 2.3442888 ; accuracy 0.13863334
epoch 10 : loss 2.3096693 ; accuracy 0.1515
epoch 11 : loss 2.276796 ; accuracy 0.16505
epoch 12 : loss 2.2453485 ; accuracy 0.1801
epoch 13 : loss 2.215092 ; accuracy 0.19401667
epoch 14 : loss 2.185849 ; accuracy 0.20971666
epoch 15 : loss 2.1574829 ; accuracy 0.22528334
epoch 16 : loss 2.1298892 ; accuracy 0.24068333
epoch 17 : loss 2.1029925 ; accuracy 0.25576666
epoch 18 : loss 2.0767288 ; accuracy 0.27163333
epoch 19 : loss 2.0510502 ; accuracy 0.28813332
epoch 20 : loss 2.0259228 ; accuracy 0.30391666
epoch 21 : loss 2.001316 ; accur